<h1>SQLAlchemy Tutorial<h1/>

In [1]:
import sqlalchemy

In [2]:
sqlalchemy.__version__

'2.0.45'

In [3]:
from sqlalchemy import create_engine, text

In [4]:
engine = create_engine("sqlite+pysqlite:///:memory:", echo=True)

In [5]:
with engine.connect() as conn:
    result =  conn.execute(text("select 'hello world'"))
    print(result.all())

2026-01-23 23:43:24,890 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 23:43:24,893 INFO sqlalchemy.engine.Engine select 'hello world'
2026-01-23 23:43:24,895 INFO sqlalchemy.engine.Engine [generated in 0.00491s] ()
[('hello world',)]
2026-01-23 23:43:24,898 INFO sqlalchemy.engine.Engine ROLLBACK


In [6]:
# Commit as you go"
with engine.connect() as conn:
    conn.execute(text("CREATE TABLE some_table (x int, y int)"))
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"), 
        [{"x": 1, "y": 1}, {"x": 2, "y": 4}],
    )
    conn.commit()

2026-01-23 23:43:24,921 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 23:43:24,923 INFO sqlalchemy.engine.Engine CREATE TABLE some_table (x int, y int)
2026-01-23 23:43:24,923 INFO sqlalchemy.engine.Engine [generated in 0.00234s] ()
2026-01-23 23:43:24,925 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-23 23:43:24,926 INFO sqlalchemy.engine.Engine [generated in 0.00092s] [(1, 1), (2, 4)]
2026-01-23 23:43:24,928 INFO sqlalchemy.engine.Engine COMMIT


In [7]:
# begins once#
with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"),
        [{"x": 6, "y": 8}, {"x": 9, "y": 10}],
    )

2026-01-23 23:43:24,953 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 23:43:24,954 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-23 23:43:24,955 INFO sqlalchemy.engine.Engine [cached since 0.02979s ago] [(6, 8), (9, 10)]
2026-01-23 23:43:24,958 INFO sqlalchemy.engine.Engine COMMIT


In [8]:
# begins once#
with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (9, 10)")
            )

2026-01-23 23:43:24,970 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 23:43:24,972 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (9, 10)
2026-01-23 23:43:24,973 INFO sqlalchemy.engine.Engine [generated in 0.00138s] ()
2026-01-23 23:43:24,975 INFO sqlalchemy.engine.Engine COMMIT


In [9]:
# select statement#
with engine.begin() as conn:
    query_result = conn.execute(text("SELECT * FROM some_table"))
    print(query_result.all())

2026-01-23 23:43:25,000 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 23:43:25,001 INFO sqlalchemy.engine.Engine SELECT * FROM some_table
2026-01-23 23:43:25,003 INFO sqlalchemy.engine.Engine [generated in 0.00135s] ()
[(1, 1), (2, 4), (6, 8), (9, 10), (9, 10)]
2026-01-23 23:43:25,005 INFO sqlalchemy.engine.Engine COMMIT


In [10]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y FROM some_table"))
    for row in result:
        print(f"x: {row.x} y: {row.y}")

2026-01-23 23:43:25,032 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 23:43:25,033 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table
2026-01-23 23:43:25,034 INFO sqlalchemy.engine.Engine [generated in 0.00236s] ()
x: 1 y: 1
x: 2 y: 4
x: 6 y: 8
x: 9 y: 10
x: 9 y: 10
2026-01-23 23:43:25,036 INFO sqlalchemy.engine.Engine ROLLBACK


<h2>Sending Parameters<h2/>

In [11]:
# return value of y where its value is greater than a specific value)

with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y FROM some_table WHERE y > :y"), {"y": 8})
    for row in result:
        print(f"x: {row.x} y: {row.y}")

2026-01-23 23:43:25,729 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 23:43:25,730 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table WHERE y > ?
2026-01-23 23:43:25,731 INFO sqlalchemy.engine.Engine [generated in 0.00214s] (8,)
x: 9 y: 10
x: 9 y: 10
2026-01-23 23:43:25,733 INFO sqlalchemy.engine.Engine ROLLBACK


<h2>Sending Multiple Parameters<h2/>

In [12]:
# inserting multiple records in a sql statement

with engine.connect() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"), 
        [{"x": 11, "y": 12}, {"x": 13, "y": 14}]
    )
    conn.commit()

2026-01-23 23:43:26,299 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 23:43:26,300 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-23 23:43:26,301 INFO sqlalchemy.engine.Engine [cached since 1.375s ago] [(11, 12), (13, 14)]
2026-01-23 23:43:26,302 INFO sqlalchemy.engine.Engine COMMIT


<h2>Executing with an ORM Session<h2/>

In [13]:
from sqlalchemy.orm import Session

In [14]:
stmt = text("SELECT x, y FROM some_table WHERE y > :y ORDER BY x, y")
with Session(engine) as session:
    result = session.execute(stmt, {"y": 6})
    for row in result:
        print(f"x: {row.x}, y: {row.y}")

2026-01-23 23:43:26,903 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 23:43:26,905 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table WHERE y > ? ORDER BY x, y
2026-01-23 23:43:26,906 INFO sqlalchemy.engine.Engine [generated in 0.00130s] (6,)
x: 6, y: 8
x: 9, y: 10
x: 9, y: 10
x: 11, y: 12
x: 13, y: 14
2026-01-23 23:43:26,908 INFO sqlalchemy.engine.Engine ROLLBACK


In [15]:
# commit #

with Session(engine) as session:
    result = session.execute(
        text("UPDATE some_table SET y=:y WHERE x=:x"),
        [{"x": 9, "y": 11},{"x": 13, "y": 15}],
    )
    session.commit()

2026-01-23 23:43:27,467 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 23:43:27,469 INFO sqlalchemy.engine.Engine UPDATE some_table SET y=? WHERE x=?
2026-01-23 23:43:27,470 INFO sqlalchemy.engine.Engine [generated in 0.00126s] [(11, 9), (15, 13)]
2026-01-23 23:43:27,472 INFO sqlalchemy.engine.Engine COMMIT


In [16]:
with Session(engine) as session:
    result = session.execute(
        text("UPDATE some_table SET y=:y WHERE x=:x"),
        [{"x": 9, "y": 11}, {"x": 13, "y": 15}]
    )
    session.commit()

2026-01-23 23:43:27,988 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 23:43:27,989 INFO sqlalchemy.engine.Engine UPDATE some_table SET y=? WHERE x=?
2026-01-23 23:43:27,989 INFO sqlalchemy.engine.Engine [cached since 0.5204s ago] [(11, 9), (15, 13)]
2026-01-23 23:43:27,990 INFO sqlalchemy.engine.Engine COMMIT


<h2>Setting up MetaData with Table objects<h2/>

In [ ]:
from sqlalchemy import MetaData
metadata_ob

In [ ]:
from sqlalchemy import Table, Column, Integer, String
user_table = Table(
    "user_account",
    metadata_obj,
    Column("id", Interger, primary_key=True),
    Column("name", String(30)),
    Column("fullname", String),
)